# CS-546 Final Project

## Authors: Tobi Kohn & Trent VanHawkins

### Introduction
Pathogens displaying Long-Distance Dispersal (LDD) pose a substantial threat to global food security and agronomics. Agricultural managemenet practices exist often include treatments such as fungicide, herbicide, or pesticid; however,  airborne pathogens can readily invade across abritrary jurisdictional boundaries and propogate new infections where management practices differ. Anually, these diseases poses an estimated $220 Billion in harvest loss [CITE].  

The Willamette Valley is a global leader in the production of Hops (*Humulus lupulus*). Powdery Mildew (*Podosphaera macularis*) is a regionally important diesease of hops, and can lead to total crop loss if not managed appropriately [CITE]. The disease appeared in the Willamette Valley for the first time in 1998, and has occured in every subsequent year up until the present. 

Models of disease spread are a classical focus of pathology research. In recent years, however, these classical models have been placed into the context of spatially explicit networks, which use stochastic models to learn network structures across discrete 

### Methods
#### The Data
#### How we learned the network structure
#### How we estimated the edge-weights
#### Estimating source-strength (outward degree) and an investigation of its properties
#### Analysis of source-strength stratified by yard-type and grower characteristics (sprays & pruning)
#### 

### Results
#### Source Strength behaviors
#### Soruce stregnth stratified analysis
#### Centrality analysis

### Discussion
#### General summary of conclusions
#### Limitations
#### Future directions

In [1]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import networkx as nx

# Read in the adjacency matrices from the CSV files
apr_may = pd.read_csv('../reports/networks/network_may_jun.csv', index_col=0, header=0)
may_jun = pd.read_csv('../reports/networks/network_may_jun.csv', index_col=0, header=0)
jun_jul = pd.read_csv('../reports/networks/network_jun_jul.csv', index_col=0, header=0)

#Read in the metadata
meta = pd.read_csv('../data/processed/clean_summary.csv', index_col=0, header=0)
meta["date"] = pd.to_datetime(meta["date"])

# Subset to 2017
meta_2017 = meta.loc[meta["year"] == 2017, ["field_id", "centroid_lat", "centroid_long", "initial_strain", "plant_type"]]
meta_2017.drop_duplicates(subset="field_id", keep="first", inplace=True)

#Write a function to build an igraph with metadata
def build_graph(adj_df, meta_df):
    # Build graph
    g = ig.Graph.Weighted_Adjacency(adj_df.to_numpy().transpose(), mode="directed", attr="weight", loops=False)
    
    # Add node names
    g.vs["name"] = adj_df.columns.tolist()
    
    # Add metadata attributes
    meta_indexed = meta_df.set_index("field_id")
    for col in meta_indexed.columns:
        g.vs[col] = [meta_indexed.loc[int(v["name"]), col] for v in g.vs]
    
    return g

apr_may_g = build_graph(apr_may, meta_2017)
may_jun_g = build_graph(may_jun, meta_2017)
jun_jul_g = build_graph(jun_jul, meta_2017)


In [2]:
out_apr_may = apr_may_g.strength(mode = "out", weights="weight")
out_may_jun = may_jun_g.strength(mode = "out", weights="weight")
out_jun_jul = jun_jul_g.strength(mode = "out", weights="weight")

In [ ]:
def plot_network(g, title="", weight_percentile=75):
    
    # Filter edges below percentile threshold
    weights = [e["weight"] for e in g.es]
    threshold = np.percentile(weights, weight_percentile)
    g_filtered = g.copy()
    g_filtered.delete_edges([e.index for e in g_filtered.es if e["weight"] < threshold])

    # Remove isolated nodes (optional)
    g_filtered.delete_vertices([v.index for v in g_filtered.vs if g_filtered.degree(v.index) == 0])

    # Node properties
    pos = {v["name"]: (v["centroid_long"], v["centroid_lat"]) for v in g_filtered.vs}
    node_colors = ["crimson" if v["plant_type"] == "R6" else "steelblue" for v in g_filtered.vs]
    weighted_outdegree = g_filtered.strength(mode="out", weights="weight")
    # Normalise to a size range
    min_size, max_size = 100, 1000
    w_min, w_max = min(weighted_outdegree), max(weighted_outdegree)
    node_sizes = [min_size + (w - w_min) / (w_max - w_min + 1e-9) * (max_size - min_size) 
                for w in weighted_outdegree]

    # Edge properties
    edge_weights = [e["weight"] for e in g_filtered.es]
    w_min, w_max = min(edge_weights), max(edge_weights)
    edge_colors = [(0, 0, 0, (e["weight"] - w_min) / (w_max - w_min + 1e-9)) for e in g_filtered.es]  

    # Convert to networkx for plotting
    G = nx.DiGraph()
    for v in g_filtered.vs:
        G.add_node(v["name"])
    for e in g_filtered.es:
        G.add_edge(g_filtered.vs[e.source]["name"], g_filtered.vs[e.target]["name"], weight=e["weight"])

    fig, ax = plt.subplots(figsize=(12, 10))

    nx.draw_networkx_nodes(G, pos=pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.75)
    nx.draw_networkx_labels(G, pos=pos, ax=ax, font_size=6, font_color="white")
    nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color=edge_colors,
                           arrows=True, arrowsize=15, width=1.5,
                           connectionstyle="arc3,rad=0.1")  # slight curve to show directionality

    # Basemap
    ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.Voyager)

    # Legend
    from matplotlib.lines import Line2D
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson", markersize=10, label="R6"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=10, label="Non-R6")]
    ax.legend(handles=legend, loc="upper left")

    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_network(apr_may_g, title="Apr–May", weight_percentile=50)
plot_network(may_jun_g, title="May–Jun", weight_percentile=50)
plot_network(jun_jul_g, title="Jun–Jul", weight_percentile=50)